In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mabs_funcs as mf

import matplotlib.colors as colors
import matplotlib.cm as cm

import matplotlib.patches as patches
import itertools
from matplotlib.ticker import LogLocator, ScalarFormatter

In [ ]:
kper, kpara, pk = np.load('/home/sukhdeep/Documents/Research/Bispectrum/3D_visibility_bispectrum/on_data/asif_da/mwa_pk/pkdata.npy', allow_pickle=True)

plt.plot(abs(pk[8]))
plt.yscale('log')

In [ ]:

import numpy as np

# Given inputs:
# kper: 1D array of shape (20,)
# kpara: 1D array of shape (668,)
# pk: 2D array of shape (20, 668)
# Choose number of bins for k
n_bins = 20  # adjust as needed

# Compute combined k grid
k2D = np.sqrt(kper[:, None]**2 + kpara[None, :]**2)

# Flatten arrays for binning
k_flat = k2D.ravel()
pk_flat = pk.ravel()

# Exclude zero or negative k values for log binning
positive_mask = k_flat > 0
kpos = k_flat[positive_mask]
pkpos = pk_flat[positive_mask]

# Define log-spaced bin edges and centers
k_min, k_max = kpos.min(), kpos.max()
bin_edges = np.logspace(np.log10(k_min), np.log10(k_max), n_bins + 1)
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean centers

# Digitize k values into bins
bin_indices = np.digitize(kpos, bin_edges)

# Initialize array to hold binned pk
pk_binned = np.empty(n_bins)
pk_count = np.zeros(n_bins, dtype=int)

# Compute mean pk in each log-bin
for i in range(1, n_bins + 1):
    mask = bin_indices == i
    pk_count[i-1] = mask.sum()
    pk_binned[i-1] = pkpos[mask].mean() if pk_count[i-1] > 0 else np.nan

fact = (bin_centers**3) / (2*np.pi*np.pi)
# Plotting k vs. P(k)
plt.figure()
plt.loglog(bin_centers, fact*np.abs(pk_binned),'-o')
plt.xlabel('k')
plt.ylabel('P(k)')
plt.title('Binned Power Spectrum')
plt.grid(True)
plt.show()


In [ ]:
output_filename = 'binned_pk_19.npz'
np.savez(output_filename,
         bin_centers,
         pk_binned)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

nbins = 50
hist_dat_filtered = pk[pk != 0]  # Keep both negative and positive values

# Define symmetric bins (log-spaced in magnitude, signed)
mag = np.abs(hist_dat_filtered)
xmin, xmax = mag.min(), mag.max()
log_bins = np.logspace(np.log10(xmin), np.log10(xmax), nbins)
bins = np.concatenate((-log_bins[::-1], log_bins))  # Include negative side

# Plot histogram
_ = plt.hist(hist_dat_filtered, bins=bins)

# Set symlog scale for x and log scale for y
plt.xscale('symlog', linthresh=1e1)  # Adjust linthresh as needed
plt.yscale('log')

plt.xlabel('cylbs (symlog scale)')
plt.ylabel('Counts (log scale)')
plt.tight_layout()
plt.show()
